In [ ]:
!pip install qiskit --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.8/9.8 MB 64.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 58.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.5/54.5 kB 3.5 MB/s eta 0:00:00


In [ ]:
pip install qiskit-ibm-runtime


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 32.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 400.4/400.4 kB 22.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.3/111.3 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 224.2/224.2 kB 14.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.4/76.4 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 130.2/130.2 kB 7.0 MB/s eta 0:00:00


In [ ]:
#!/usr/bin/env python3
"""
ASI Cuántica PURA — IBM Quantum REAL HARDWARE
qiskit_ibm_runtime SamplerV2
"""
import os, numpy as np
from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister, transpile
from qiskit_ibm_runtime import QiskitRuntimeService, SamplerV2 as Sampler

# --- TU TOKEN ---
# export IBM_QUANTUM_TOKEN="tu_token_aqui"
# o ponelo directo (NO subir a git):
TOKEN = os.getenv("IBM_QUANTUM_TOKEN", "your_token_here")

# --- construir ASI idéntico al ultra realista ---
def construir_asi(alpha, beta, n_grover=1):
    qc_reg = QuantumRegister(2, 'C')
    qi = QuantumRegister(2, 'I')
    cr = ClassicalRegister(2, 'c')
    qc = QuantumCircuit(qc_reg, qi, cr, name='ASI_Real')
    for q in qc_reg: qc.h(q)
    qc.barrier()
    qc.ry(alpha, qi[0]); qc.ry(beta, qi[1]); qc.barrier()
    clases = [(0.0,0.0),(np.pi/2,0.0),(0.0,np.pi/2),(np.pi/2,np.pi/2)]
    psi = np.array([np.cos(alpha/2)*np.cos(beta/2),
                    np.cos(alpha/2)*np.sin(beta/2),
                    np.sin(alpha/2)*np.cos(beta/2),
                    np.sin(alpha/2)*np.sin(beta/2)], dtype=complex)
    for k,(tk,fk) in enumerate(clases):
        ck = np.array([np.cos(tk/2)*np.cos(fk/2),
                       np.cos(tk/2)*np.sin(fk/2),
                       np.sin(tk/2)*np.cos(fk/2),
                       np.sin(tk/2)*np.sin(fk/2)], dtype=complex)
        sim = abs(np.vdot(psi,ck))**2
        phase = np.pi * sim**3
        bits = format(k,'02b')
        for i,b in enumerate(bits):
            if b=='0': qc.x(qc_reg[i])
        qc.cp(phase, qc_reg[0], qc_reg[1])
        for i,b in enumerate(bits):
            if b=='0': qc.x(qc_reg[i])
        qc.barrier()
    # Grover
    for _ in range(n_grover):
        for q in qc_reg: qc.h(q); qc.x(q)
        qc.cz(qc_reg[0], qc_reg[1])
        for q in qc_reg: qc.x(q); qc.h(q)
        qc.barrier()
    for i in range(2): qc.cx(qc_reg[i], qi[i])
    qc.barrier()
    qc.measure(qc_reg[0], cr[0])
    qc.measure(qc_reg[1], cr[1])
    return qc

# --- conectar IBM ---
# Nuevo IBM Quantum Platform:
# service = QiskitRuntimeService(channel="ibm_quantum_platform", token=TOKEN)
# Legacy:
# service = QiskitRuntimeService(channel="ibm_quantum", token=TOKEN, instance="ibm-q/open/main")

print("Conectando IBM Quantum...")
try:
    service = QiskitRuntimeService(channel="ibm_quantum_platform", token=TOKEN)
except Exception:
    service = QiskitRuntimeService(channel="ibm_quantum", token=TOKEN)

# backend: el menos ocupado Heron
backend = service.least_busy(operational=True, simulator=False, min_num_qubits=5)
print(f"Backend real: {backend.name}  qubits={backend.num_qubits}")

# pruebas zero-shot
pruebas = [
    ("C0",0.10,0.10),
    ("C1",np.pi/2,0.05),
    ("C2",0.05,np.pi/2),
    ("C3",np.pi/2,np.pi/2),
    ("amb1",1.5,0.7),
    ("amb2",1.0,1.0),
]
circuits = [construir_asi(a,b) for _,a,b in pruebas]
# transpile para backend real
tcircs = transpile(circuits, backend=backend, optimization_level=3)

sampler = Sampler(mode=backend)
print(f"Enviando {len(tcircs)} circuitos... shots=20000")
job = sampler.run(tcircs, shots=20000)
print(f"Job ID: {job.job_id()}")
print("Esperando resultado (cola Heron suele ser 5-90 min)...")
result = job.result()
print("\n=== RESULTADOS IBM QUANTUM REAL ===")
clases = ['00','01','10','11']
for i,(nombre,_,_) in enumerate(pruebas):
    quasi = result[i].data.c
    # quasi_dist: convertir a counts
    counts = quasi.get_int_counts()
    total = sum(counts.values())
    # mapear int -> bitstring c1c0
    dist = {f"{k:02b}":v/total for k,v in counts.items()}
    # nuestro análisis sin reverse (key directo)
    winner = max(dist, key=dist.get)
    print(f"{nombre:6} → |{winner}⟩ {dist[winner]*100:.1f}%  {dist}")
print("\nListo. Revoca tu token en quantum.ibm.com")


qiskit_runtime_service._discover_account:WARNING:2026-07-07 14:34:42,770: Loading account with the given token. A saved account will not be used.


Conectando IBM Quantum...


qiskit_runtime_service.__init__:WARNING:2026-07-07 14:34:45,823: Instance was not set at service instantiation. Free and trial plan instances will be prioritized. Based on the following filters: (tags: None, region: us-east, eu-de), and available plans: (open), the available account instances are: open-instance. If you need a specific instance set it explicitly either by using a saved account with a saved default instance or passing it in directly to QiskitRuntimeService().
qiskit_runtime_service.backends:WARNING:2026-07-07 14:34:46,175: Loading instance: open-instance, plan: open
qiskit_runtime_service.backends:WARNING:2026-07-07 14:34:49,764: Using instance: open-instance, plan: open


Backend real: ibm_marrakesh  qubits=156
Enviando 6 circuitos... shots=20000
Job ID: d96gt2t2su3c739gueqg
Esperando resultado (cola Heron suele ser 5-90 min)...

=== RESULTADOS IBM QUANTUM REAL ===
C0     → |00⟩ 90.2%  {'00': 0.90195, '11': 0.06425, '01': 0.0149, '10': 0.0189}
C1     → |10⟩ 91.4%  {'10': 0.91365, '11': 0.01455, '01': 0.05285, '00': 0.01895}
C2     → |01⟩ 91.5%  {'10': 0.0468, '01': 0.91495, '11': 0.01515, '00': 0.0231}
C3     → |11⟩ 92.1%  {'11': 0.92085, '01': 0.01815, '00': 0.04075, '10': 0.02025}
amb1   → |10⟩ 38.7%  {'10': 0.387, '01': 0.25315, '00': 0.192, '11': 0.16785}
amb2   → |11⟩ 38.1%  {'01': 0.1969, '11': 0.38105, '00': 0.2607, '10': 0.16135}

Listo. Revoca tu token en quantum.ibm.com
